# Choosing eps From the k-Distance Curve

**Topic:** DBSCAN &nbsp;·&nbsp; **Difficulty:** Medium

DBSCAN's `eps` is the parameter everyone guesses and everyone gets wrong. Too
small and everything is noise. Too large and the whole dataset is one cluster.

There is a standard way to pick it. Sort every point by the distance to its
`min_samples`-th nearest neighbour, plot that curve, and look for the knee: the
point where distances stop being typical and start being outliers. Find the knee
numerically and use it.

## The data (created in the next cell)

- `X` `(700, 2)`: two crescents plus 60 uniform noise points.
- `min_samples = 8`.

## Your task

With **sklearn and NumPy**:

1. Compute each point's distance to its 7th nearest neighbour (excluding itself)
   and sort those distances ascending. Call it `kdist`.
2. Find the knee as the point of maximum distance from the chord joining the
   first and last points of the sorted curve, a standard geometric knee finder.
   Use the index of that maximum, and set `eps_knee = kdist[knee_index]`.
3. Run `DBSCAN(eps=eps_knee, min_samples=8)`.
4. Set `answers` to a dict with `"eps_knee"`, `"n_clusters"` (int, excluding the
   noise label `-1`), `"n_noise"` (int), and `"ari"`, the adjusted Rand index
   against `y_true` computed on the **non-noise true points only**.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks you found exactly 2 clusters and scored well on the crescents.

In [ ]:
import numpy as np

rng = np.random.default_rng(2020)
t = rng.uniform(0, np.pi, 320)
c1 = np.column_stack([np.cos(t), np.sin(t)]) + rng.normal(0, 0.06, (320, 2))
c2 = np.column_stack([1 - np.cos(t), 0.5 - np.sin(t)]) + rng.normal(0, 0.06, (320, 2))
noise = rng.uniform([-1.6, -1.1], [2.6, 1.6], (60, 2))
X = np.vstack([c1, c2, noise])
y_true = np.concatenate([np.zeros(320, int), np.ones(320, int),
                         np.full(60, -1, int)])
min_samples = 8
print(f"X {X.shape}: two crescents plus {len(noise)} noise points")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("eps_knee", [], 0.1455333005644029, -0.02436780693590545, 1e-06),
    ("n_clusters", [], 2.0, -0.3348760296289983, 1e-06),
    ("n_noise", [], 45.0, -7.534710666652462, 1e-06),
    ("ari", [], 0.9937499237531444, -0.1663915144552864, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "n_clusters" in answers and "ari" in answers:
    _g = int(answers["n_clusters"]) == 2 and answers["ari"] > 0.9
    print(f"  {'two crescents recovered':<24} {'ok' if _g else 'wrong'}  "
          f"({int(answers['n_clusters'])} clusters, ARI {float(answers['ari']):.3f})")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

The k-distance curve is flat while you are inside a dense region and turns sharply
upward once you reach the sparse points, so the knee separates "typical
neighbourhood scale" from "outlier". The chord method is the cheapest reliable
way to locate it: the knee is the point furthest from the straight line joining
the curve's endpoints. Unlike k-means, DBSCAN is never forced to assign the noise
points. They come back as `-1`, which is the entire reason to use it here.

```python
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.metrics import adjusted_rand_score

D = np.sqrt(((X[:, None, :] - X[None, :, :]) ** 2).sum(axis=2))
np.fill_diagonal(D, np.inf)
kdist = np.sort(np.sort(D, axis=1)[:, min_samples - 2])

n = len(kdist)
x = np.arange(n, dtype=float)
p0 = np.array([x[0], kdist[0]])
p1 = np.array([x[-1], kdist[-1]])
d = p1 - p0
d = d / np.linalg.norm(d)
pts = np.column_stack([x, kdist]) - p0
proj = pts @ d
perp = np.linalg.norm(pts - proj[:, None] * d[None, :], axis=1)
knee = int(np.argmax(perp))
eps_knee = float(kdist[knee])

lab = DBSCAN(eps=eps_knee, min_samples=min_samples).fit_predict(X)
real = y_true != -1

answers = {
    "eps_knee": eps_knee,
    "n_clusters": int(len(set(lab.tolist()) - {-1})),
    "n_noise": int((lab == -1).sum()),
    "ari": float(adjusted_rand_score(y_true[real], lab[real])),
}
```